In [1]:
import pandas as pd
import numpy as np

# Mapping of questions to learning styles
question_styles = {'Q0':'K', 'Q1':'V', 'Q2':'R', 'Q3':'R','Q4':'K', 'Q5':'A', 'Q6':'K', 'Q7':'V', 'Q8':'K', 'Q9':'K', 'Q10':'K', 'Q11':'K', 'Q12': 'K', 'Q13':'K', 'Q14':'V',
        'Q15':'V','Q16': 'V', 'Q17':'R','Q18': 'V', 'Q19': 'V', 'Q20': 'V',
        'Q21':'A','Q22': 'A', 'Q23':'A','Q24':'A', 'Q25':'K','Q26':'V','Q27':'K'}

# Mapping of responses to scores
prob_dist = {'SA': 1.00, 'A': 0.75, 'N': 0.50, 'D': 0.25, 'SD': 0.00}
possible_responses = list(prob_dist.keys())

# Number of students
num_students = 800

# Probability of a missing response for each question
missing_prob = 0.05

# Probability distribution for responses based on preference
preferred_dist = {'SA': 0.4, 'A': 0.4, 'N': 0.1, 'D': 0.05, 'SD': 0.05}
non_preferred_dist = {'SA': 0.05, 'A': 0.15, 'N': 0.4, 'D': 0.25, 'SD': 0.15}
default_dist = {'SA': 0.2, 'A': 0.3, 'N': 0.3, 'D': 0.15, 'SD': 0.05} # In case no clear preference

# Create an empty list to store student data
student_data = []

# Generate data for each student
for i in range(num_students):
    student_responses = {}
    vark_scores = {'V': 0, 'R': 0, 'A': 0, 'K': 0}

    # Determine student's learning style preferences (can be multiple)
    # preferences = np.random.choice(['V', 'R', 'A', 'K'], size=np.random.randint(1, 4), replace=False)
    preferences = np.random.choice(['V', 'R', 'A', 'K'], size=np.random.choice([1, 2, 3], p=[0.5, 0.3, 0.2]), replace=False)
    student_preferences = list(preferences)

    # Generate responses for each question with bias
    for q in range(28):
        question_id = f'Q{q}'
        question_style = question_styles[question_id]

        if np.random.rand() < missing_prob:
            student_responses[question_id] = np.nan
        else:
            if question_style in student_preferences:
                response = np.random.choice(possible_responses, p=list(preferred_dist.values()))
            else:
                response = np.random.choice(possible_responses, p=list(non_preferred_dist.values()))
            student_responses[question_id] = response
            vark_scores[question_style] += prob_dist[response]

    # Determine dominant learning style(s) based on scores
    max_score = max(vark_scores.values())
    dominant_styles = [style for style, score in vark_scores.items() if score == max_score]

    # If there are ties, consider all tied styles as preferences
    if len(dominant_styles) > 1:
        preferred_style = ', '.join(dominant_styles)
    elif len(dominant_styles) == 1:
        preferred_style = dominant_styles[0]
    else:
        preferred_style = 'Undetermined' # In case all scores are zero

    # Add student data to the list
    student_data.append({
        'StudentID': i + 1,
        'Preferences': ', '.join(student_preferences),
        **student_responses,
        'V_Score': vark_scores['V'],
        'R_Score': vark_scores['R'],
        'A_Score': vark_scores['A'],
        'K_Score': vark_scores['K'],
        'Preferred_Style': preferred_style
    })

# Create a pandas DataFrame from the collected data
df = pd.DataFrame(student_data)

# Display the first few rows of the DataFrame
print(df.head())

# Provide code to download the DataFrame as a CSV file
csv_filename = 'student_learning_styles_biased.csv'
# df.to_csv(csv_filename, index=False)
print(f"\nDataset successfully created and saved as '{csv_filename}' with biased responses and potential multiple preferences.")

   StudentID Preferences  Q0   Q1  Q2  Q3   Q4  Q5  Q6   Q7  ...  Q23 Q24  \
0          1        V, A  SD  NaN   N   D    A  SA  SD   SA  ...    A   A   
1          2        K, A   A    N   D   N    N   A  SA    D  ...   SA   A   
2          3           R  SD    N   A   A  NaN  SD  SD  NaN  ...    N  SD   
3          4     A, R, V  SA  NaN   A   N    D   A   N   SA  ...    A  SA   
4          5           A   N    A  SD  SD   SD  SA   N   SD  ...  NaN  SA   

   Q25 Q26  Q27 V_Score R_Score A_Score K_Score Preferred_Style  
0  NaN   A   SD    6.75    1.00    4.25    3.25               V  
1   SA   N   SA    3.00    1.25    4.25    8.50               K  
2    D   N  NaN    1.75    2.25    1.50    2.50               K  
3    N  SA    D    5.50    2.25    4.50    6.00               K  
4    N   N   SD    3.50    0.25    3.75    4.50               K  

[5 rows x 35 columns]

Dataset successfully created and saved as 'student_learning_styles_biased.csv' with biased responses and potential mu

In [32]:
df.columns

Index(['StudentID', 'Preferences', 'Q0', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6',
       'Q7', 'Q8', 'Q9', 'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16',
       'Q17', 'Q18', 'Q19', 'Q20', 'Q21', 'Q22', 'Q23', 'Q24', 'Q25', 'Q26',
       'Q27', 'V_Score', 'R_Score', 'A_Score', 'K_Score', 'Preferred_Style',
       'Visual', 'Auditorial', 'Reading', 'Kinesthetic'],
      dtype='object')

In [2]:
df.fillna('N', inplace=True)
Prob_dist = {'SA': 1.00, 'A': 0.75, 'N': 0.50, 'D': 0.25, 'SD': 0.00}

# Apply the replacement only to the columns 'Q0' to 'Q27'
columns_to_replace = [f'Q{i}' for i in range(28)]
df[columns_to_replace] = df[columns_to_replace].replace(Prob_dist)

df.rename(columns={'V_Score': 'Visual', 'R_Score': 'Reading', 'A_Score': 'Auditorial', 'K_Score': 'Kinesthetic'}, inplace = True)

LSQ = {'Q0':'K', 'Q1':'V', 'Q2':'R', 'Q3':'R','Q4':'K', 'Q5':'A', 'Q6':'K', 'Q7':'V', 'Q8':'K', 'Q9':'K', 'Q10':'K', 'Q11':'K', 'Q12': 'K', 'Q13':'K', 'Q14':'V',
        'Q15':'V','Q16': 'V', 'Q17':'R','Q18': 'V', 'Q19': 'V', 'Q20': 'V',
        'Q21':'A','Q22': 'A', 'Q23':'A','Q24':'A', 'Q25':'K','Q26':'V','Q27':'K'}

Prob_dist = {'SA': 1.00, 'A': 0.75, 'N': 0.50, 'D': 0.25, 'SD': 0.00}

question_weights = {
    "Visual": [key for key, val in LSQ.items() if val == 'V'],
    "Auditorial": [key for key, val in LSQ.items() if val == 'A'],
    "Reading": [key for key, val in LSQ.items() if val == 'R'],
    "Kinesthetic": [key for key, val in LSQ.items() if val == 'K']
}

df[['Visual', 'Auditorial', 'Reading', 'Kinesthetic']] = 0

for student in range(df.shape[0]):
    for style in question_weights:
        scores = [df.loc[student, Q] for Q in question_weights[style]]
        prob = sum(scores)/len(scores)
        df.loc[student, style] = prob

C:\Users\Tanima\AppData\Local\Temp\ipykernel_3308\71062300.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[columns_to_replace] = df[columns_to_replace].replace(Prob_dist)
C:\Users\Tanima\AppData\Local\Temp\ipykernel_3308\71062300.py:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.8055555555555556' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[student, style] = prob
C:\Users\Tanima\AppData\Local\Temp\ipykernel_3308\71062300.py:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.85' has dtype incompatible with int64, please explicitly ca

In [42]:
df

,StudentID,Preferences,Q0,Q1,Q2,Q3,Q4,Q5,Q6,Q7,...,Q23,Q24,Q25,Q26,Q27,Visual,Reading,Auditorial,Kinesthetic,Preferred_Style
0,1,"A, K",1.00,0.75,0.25,1.00,0.50,0.75,0.50,0.75,...,0.75,0.75,0.75,0.75,1.00,0.527778,0.666667,0.85,0.727273,K
1,2,R,0.50,0.50,0.75,1.00,0.50,0.50,0.75,1.00,...,0.50,0.50,0.25,0.75,0.50,0.472222,0.583333,0.45,0.477273,K
2,3,"V, R, K, A",1.00,1.00,0.25,1.00,1.00,1.00,0.75,1.00,...,1.00,0.75,1.00,0.75,1.00,0.888889,0.750000,0.95,0.886364,K
3,4,"K, V, R, A",1.00,0.75,0.75,0.75,1.00,1.00,0.75,0.75,...,1.00,1.00,1.00,0.50,1.00,0.722222,0.833333,0.90,0.863636,K
4,5,K,0.50,0.50,0.25,0.50,1.00,0.00,0.75,0.25,...,0.25,0.50,1.00,0.50,0.50,0.416667,0.333333,0.30,0.772727,K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,796,"K, R, V",0.75,0.50,0.75,0.75,1.00,0.25,1.00,0.75,...,0.50,0.50,0.75,0.75,0.50,0.777778,0.750000,0.45,0.750000,K
796,797,A,0.25,0.00,0.50,0.00,0.50,0.00,0.25,0.50,...,1.00,0.75,0.25,0.50,0.25,0.444444,0.333333,0.55,0.386364,K
797,798,"V, A, R, K",1.00,0.75,0.75,0.50,0.75,1.00,1.00,1.00,...,1.00,1.00,0.25,0.75,0.75,0.750000,0.416667,0.90,0.750000,K
798,799,A,0.50,0.50,0.50,0.00,0.50,0.75,0.50,0.25,...,0.50,0.75,0.00,0.50,0.50,0.500000,0.333333,0.75,0.454545,"V, K"


In [3]:
from sklearn.model_selection import train_test_split

# Assuming your DataFrame is named 'df'

# Separate features (X) and target (y)
# X = df.drop('Preferences', axis=1)
y = df['Preferences']

# Split the data into training and testing sets with an 80% ratio, stratified by 'Preferred_Style'
train_df, test_df = train_test_split(df, test_size=0.2, random_state=2025, stratify=y)

# Display the sizes of the training and testing sets
print(f"Training set size: {len(train_df)}")
print(f"Testing set size: {len(test_df)}")

# Display the distribution of preferred styles in the training set
print("\nDistribution of Preferred Styles in Training Set:")
print(train_df['Preferred_Style'].value_counts(normalize=True))

# Display the distribution of preferred styles in the testing set
print("\nDistribution of Preferred Styles in Testing Set:")
print(test_df['Preferred_Style'].value_counts(normalize=True))

# Now you have your training data in 'train_df' and testing data in 'test_df'
# You can download these DataFrames as CSV files if needed:
train_csv_filename = 'train_student_learning_styles.csv'
test_csv_filename = 'test_student_learning_styles.csv'

train_df.to_csv(train_csv_filename, index=False)
test_df.to_csv(test_csv_filename, index=False)


Training set size: 640
Testing set size: 160

Distribution of Preferred Styles in Training Set:
Preferred_Style
K          0.609375
V          0.329688
A          0.034375
V, K       0.015625
A, K       0.006250
V, A       0.003125
V, A, K    0.001563
Name: proportion, dtype: float64

Distribution of Preferred Styles in Testing Set:
Preferred_Style
K       0.58125
V       0.34375
A       0.04375
V, K    0.03125
Name: proportion, dtype: float64


In [38]:
df['Preferences'].value_counts()

Preferences
V             61
K             47
R             42
A             38
K, R          17
              ..
V, R, A        4
A, K, R, V     4
R, V, A, K     4
R, A, V, K     4
A, V, R        3
Name: count, Length: 64, dtype: int64